# Maia Corpus v9 — Offline Rights Classification v0.11
Processamento integral offline do inventário v0.10. Não altera originais. Candidatos por datas não são autorização jurídica. Aprovação exige evidência explícita aplicável ao texto exato; o catálogo recuperado não necessariamente contém essa evidência. Nenhuma revisão manual é exigida para terminar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv,json,re,collections,os,hashlib,datetime
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
SRC=ROOT/'gutenberg_v10_hf_rights/rights_inventory_latest.csv'
OUT=ROOT/'gutenberg_v11_offline_rights'; OUT.mkdir(exist_ok=True)
csv.field_size_limit(100_000_000)
with SRC.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
assert rows and 'hf_evidence_json' in rows[0], 'Inventário incompatível'
print('Livros:',len(rows),'Colunas:',list(rows[0]))
YEAR=2026; CUTOFF=YEAR-71


## Regras e diagnóstico
Somente PG Catalog.csv é usado para autoria; estimativas temporais não são evidência de falecimento. Datas incompletas, autoria coletiva ambígua e colaboradores desconhecidos não geram aprovação. A ausência de tradutor no catálogo não prova inexistência de tradução.

In [ ]:
def norm(k):return re.sub(r'[^a-z0-9]','',str(k).lower())
def catalog(r):
    ev=json.loads(r.get('hf_evidence_json') or '[]')
    return [(e,x) for e in ev if e.get('repo')=='Texttechnologylab/project-gutenberg-temporal-corpus' and e.get('file','').endswith('/PG Catalog.csv') for x in (e['metadata'] if isinstance(e.get('metadata'),list) else [e.get('metadata',{})]) if isinstance(x,dict)]
def parse_authors(s):
    # Catálogo separa autores por ponto e vírgula; não confundir vírgula do sobrenome.
    parts=[x.strip() for x in str(s).split(';') if x.strip()]
    result=[]
    for part in parts:
        match=re.search(r'(?<!\d)(\d{3,4})\s*[-–—]\s*(\d{3,4})\s*$',part)
        death=int(match[2]) if match and int(match[1])<=int(match[2])<=YEAR else None
        result.append({'raw':part,'death_year':death})
    return result
assert parse_authors('Dickens, Charles, 1812-1870')[0]['death_year']==1870
assert parse_authors('Author, 1900-')[0]['death_year'] is None
assert len(parse_authors('A, 1800-1870; B, 1900-1980'))==2
fields=collections.Counter(); examples=[]
for r in rows:
    for e,x in catalog(r):
        fields.update(x.keys())
        if len(examples)<5:examples.append(x)
print('Campos reais do catálogo:',fields)
print(json.dumps(examples,ensure_ascii=False,indent=2)[:8000])
assert fields,'Catálogo não encontrado no inventário; não continuar com índice vazio.'


In [ ]:
def classify(r):
    records=catalog(r)
    authors=set(); titles=set(); collaborators=[]; evidence=[]
    for e,x in records:
        evidence.append({'repo':e.get('repo'),'revision':e.get('revision'),'file':e.get('file'),'metadata':x})
        for k,v in x.items():
            if not v:continue
            if norm(k) in ('author','authors','creator','creators'):authors.add(str(v).strip())
            if norm(k)=='title':titles.add(str(v))
            if norm(k) in ('translator','translators','editor','editors','contributor','contributors'):collaborators.append(str(v))
    parsed=parse_authors(next(iter(authors))) if len(authors)==1 else []
    years=[a['death_year'] for a in parsed]
    status='AUTO_QUARANTINE'; reason='missing_or_conflicting_author_dates'
    if years and all(y is not None for y in years):
        if max(years)<=CUTOFF:
            status='CANDIDATE_PUBLIC_DOMAIN';reason='all_parsed_author_dates_before_cutoff_edition_unverified'
        else:reason='author_date_after_cutoff'
    if collaborators:
        status='AUTO_QUARANTINE';reason='additional_contributors_require_evidence'
    # Não há evidência de licença da versão textual exata no esquema v0.10.
    return {**r,'v11_title':' | '.join(sorted(titles)),'v11_authors':' | '.join(sorted(authors)),
      'v11_author_details':json.dumps(parsed,ensure_ascii=False),'v11_status':status,'v11_reason':reason,
      'approved_for_training':False,'v11_evidence':json.dumps(evidence,ensure_ascii=False)}
result=[]
for i,r in enumerate(rows,1):
    result.append(classify(r))
    if i%5000==0:print('Classificados',i,'/',len(rows),flush=True)
print(collections.Counter(r['v11_status'] for r in result))


## Volume textual e integridade
Usa tamanhos registrados no inventário quando disponíveis; caso contrário lê o arquivo local indicado pelo manifesto. Não baixa textos. Estimativa de 3–5 caracteres/token, não contagem do tokenizer definitivo. A leitura integral é opcional e desligada por padrão para evitar I/O excessivo.

In [ ]:
READ_TEXT_IF_MISSING=False
VERIFY_SHA=False
for i,r in enumerate(result,1):
    chars=None
    for key in ('characters','actual_characters','char_count','chars','n_chars'):
        try:
            chars=int(r.get(key,''));break
        except (ValueError,TypeError):pass
    path=Path(r.get('path') or r.get('file_path') or '/nonexistent')
    r['v11_integrity']='not_verified'
    if (READ_TEXT_IF_MISSING and chars is None) or VERIFY_SHA:
        try:
            assert path.resolve().is_relative_to(ROOT.resolve())
            data=path.read_bytes();chars=len(data.decode('utf-8'))
            if VERIFY_SHA:r['v11_integrity']='ok' if hashlib.sha256(data).hexdigest()==r.get('sha256') else 'failed'
        except Exception:r['v11_integrity']='read_error'
    r['v11_characters']=chars
    if i%5000==0:print('Volume',i,flush=True)


In [ ]:
def write_csv(name,data):
    p=OUT/name;t=p.with_suffix('.part')
    fields=list(dict.fromkeys(k for r in result for k in r))
    with t.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(data)
    os.replace(t,p)
write_csv('rights_inventory_latest.csv',result)
candidates=[r for r in result if r['v11_status']=='CANDIDATE_PUBLIC_DOMAIN']
write_csv('public_domain_candidates.csv',candidates)
write_csv('quarantine.csv',[r for r in result if r['v11_status']=='AUTO_QUARANTINE'])
write_csv('eligible_for_training.csv',[])
summary={'books':len(result),'approved':0,'status':dict(collections.Counter(r['v11_status'] for r in result)), 'reasons':dict(collections.Counter(r['v11_reason'] for r in result)), 'by_language':{}}
for lang in ('pt','en','es'):
    group=[r for r in candidates if r.get('language')==lang]; known=[r['v11_characters'] for r in group if r['v11_characters'] is not None]; total=sum(known)
    summary['by_language'][lang]={'candidates':len(group),'known_size':len(known),'characters_known_subset':total,'estimated_tokens_known_subset':[total//5,total//3]}
p=OUT/'rights_summary_latest.json';t=p.with_suffix('.part');t.write_text(json.dumps(summary,ensure_ascii=False,indent=2));os.replace(t,p)
print(json.dumps(summary,ensure_ascii=False,indent=2));print(OUT)


## Limite da classificação
O arquivo de elegíveis permanece vazio: datas de autores não comprovam direitos da tradução ou edição. Para obter aprovação automática efetiva, será necessária uma fonte com evidência de direitos vinculada ao texto específico. Não confundir candidatos com textos liberados. Reexecutar é seguro: processamento local determinístico, saídas substituídas atomicamente por arquivo.